# SapiMouse — hand-crafted features

`session of mouse activity -> 64-d embedding`, trained so that two windows from
the same person are close and two from different people are not.

A stroke is represented by the ~60 statistics in `stroke_features()` — durations,
speed percentiles, curvature, overshoot, click dwell. The companion notebook
`sapimouse_raw.ipynb` feeds the encoder raw point sequences instead; everything
else in the two is identical, so the pair is a controlled comparison of the
representation and nothing else.

**Why this corpus.** 120 users at ~59 Hz, against Balabit's 10 users at ~9 Hz.
On Balabit the hand-crafted features beat the learned sequence encoder in four
separate comparisons, and the most likely explanation was identity count: eight
training identities is not enough to learn a representation from scratch. 120 is
a different regime, and it is also the first corpus here that can afford a
**user-disjoint** evaluation.

**Everything reported below is open-set.** Enrolment and verification users never
appear in training, and they are enrolled from one recording and verified from
another. No number in this notebook is closed-set.

In [1]:
# ---------------------------------------------------------------------------
# Runtime + paths
# ---------------------------------------------------------------------------
RUNTIME = "auto"                                    # "auto" | "local" | "colab"
DRIVE_DATA = "/content/drive/MyDrive/trace-data/sapimouse"
DRIVE_OUT  = "/content/drive/MyDrive/trace-data"
LOCAL_CANDIDATES = ("data/sapimouse", "encoder2/data/sapimouse")

import json, math, time, warnings
from dataclasses import dataclass, asdict, field, replace
from pathlib import Path

import numpy as np
import pandas as pd


def _detect_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


IN_COLAB = _detect_colab() if RUNTIME == "auto" else RUNTIME == "colab"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, OUT_ROOT = Path(DRIVE_DATA), Path(DRIVE_OUT)
else:
    ROOT = next((p for p in map(Path, LOCAL_CANDIDATES) if p.is_dir()),
                Path(LOCAL_CANDIDATES[0])).resolve()
    OUT_ROOT = ROOT.parent
OUT_ROOT.mkdir(parents=True, exist_ok=True)

USERS = sorted((p for p in ROOT.glob("user*") if p.is_dir()),
               key=lambda p: int("".join(c for c in p.name if c.isdigit())))
print(f"runtime : {'colab' if IN_COLAB else 'local'}")
print(f"data    : {ROOT}  ({len(USERS)} users)")
print(f"outputs : {OUT_ROOT}")

runtime : local
data    : /Users/albi/Documents/Projects/IBM_hackathon_2026/encoder2/data/sapimouse  (120 users)
outputs : /Users/albi/Documents/Projects/IBM_hackathon_2026/encoder2/data


## 1. Load

120 users, each with a 1-minute and a 3-minute recording — about 4 minutes of
mouse activity per person, 1.18 M events in 32 MB. That is ~1/80th of Balabit's
data *per user* but twelve times as many users, which is the trade this notebook
exists to test.

The two-sessions-per-user structure is what makes the protocol work: enrol on the
3-minute take, verify on the 1-minute take, and the comparison is cross-session by
construction.

In [3]:
# ---------------------------------------------------------------------------
# SapiMouse loader
# ---------------------------------------------------------------------------
# Three differences from Balabit that will silently corrupt everything if missed:
#
#   1. `client timestamp` is in MILLISECONDS, not seconds. Left unconverted every
#      dt is 1000x too large, every velocity 1000x too small, and the adaptive
#      pause threshold puts each sample in its own stroke.
#   2. There is no `record timestamp` column at all.
#   3. Sessions are fixed-length takes (1 min and 3 min) rather than whatever the
#      user happened to do, so session length carries no information about the
#      person -- which is a good thing, and one less confound than Balabit had.

TIME_UNIT_S = 1e-3          # client timestamp -> seconds

DTYPES = {
    "client timestamp": "float64",
    "button": "category",
    "state": "category",
    "x": "float32",
    "y": "float32",
}


def load_session(path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=DTYPES)
    df.columns = [c.strip().lower() for c in df.columns]
    df["client timestamp"] = df["client timestamp"].astype("float64") * TIME_UNIT_S
    return df


def load_corpus(users=USERS, verbose=True):
    out = {}
    for ud in users:
        # session ids MUST be globally unique. SapiMouse names files by recording
        # date, and dozens of people recorded on the same day, so `f.stem` alone
        # collides across users -- which makes two different people's recordings
        # look like the same session to every group-by and every leak check.
        sessions = {f"{ud.name}__{f.stem}": load_session(f)
                    for f in sorted(ud.glob("session_*.csv"))}
        if sessions:
            out[ud.name] = sessions
    n_s = sum(len(v) for v in out.values())
    n_e = sum(len(d) for v in out.values() for d in v.values())
    if verbose:
        per = pd.Series({u: len(v) for u, v in out.items()})
        print(f"[load] {len(out)} users, {n_s} sessions, {n_e:,} events")
        print(f"[load] sessions per user: {per.value_counts().to_dict()}")
    return out


users_raw = load_corpus()

# sanity: the unit conversion is the one mistake that is invisible downstream
_s = next(iter(next(iter(users_raw.values())).values()))
_dt = np.diff(_s["client timestamp"].to_numpy())
_dt = _dt[_dt > 0]
print(f"[check] median dt {np.median(_dt)*1000:.1f} ms  ->  {1/np.median(_dt):.1f} Hz "
      f"(expect ~60 Hz; if this reads ~0.06 Hz the ms->s conversion is missing)")

[load] 120 users, 245 sessions, 1,184,434 events
[load] sessions per user: {2: 117, 4: 2, 3: 1}
[check] median dt 17.0 ms  ->  58.8 Hz (expect ~60 Hz; if this reads ~0.06 Hz the ms->s conversion is missing)


## 2. Clean, segment, describe

Lifted unchanged from the Balabit notebook, because none of it is corpus-specific:
event cleaning (drop scroll, out-of-range, backwards clocks, tied timestamps,
despike), stroke segmentation on pauses and clicks, and the ~60 per-stroke
features.

The knobs do change. At 59 Hz a 0.5 s pause is 30 samples rather than 4, so
`pause_split_s` drops to 0.30 and `min_points` rises to 10 — at Balabit's rate 6
points was 0.7 s of movement, here it would be 0.1 s, which is noise rather than a
stroke. `max_dt_median_s=0.10` in the QC rejects any session that is not actually
high-rate.

In [5]:
from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------


@dataclass
class Config:
    # --- which clock to trust -------------------------------------------------
    time_col: str = "client timestamp"   # or "record timestamp"

    # --- stroke segmentation --------------------------------------------------
    pause_split_s: float = 0.50          # gap that ends a stroke
    adaptive_pause: bool = True          # override with k * median(dt) if larger
    adaptive_pause_mult: float = 4.0
    split_on_click: bool = True          # a click always terminates a stroke
    max_duration_s: float = 10.0         # hard cap so one stroke can't run away
    max_points: int = 300

    # --- stroke validity ------------------------------------------------------
    min_points: int = 6
    min_path_px: float = 20.0
    min_duration_s: float = 0.05

    # --- feature knobs --------------------------------------------------------
    dir_change_deg: float = 20.0         # angle change that counts as a reversal
    stationary_px: float = 2.0           # segment shorter than this == micro-pause
    tail_frac: float = 0.25              # "final approach" = last 25% of the stroke

    # --- normalisation --------------------------------------------------------
    screen_w: float | None = None        # e.g. 1920 -> positions/lengths in screen units
    screen_h: float | None = None

    # --- misc -----------------------------------------------------------------
    verbose: bool = True


MOVE_STATES = {"move", "drag"}
DOWN_STATES = {"pressed", "down"}
UP_STATES = {"released", "up"}


# ----------------------------------------------------------------------------
# small numeric helpers
# ----------------------------------------------------------------------------


def _safe_div(a, b, fill=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full(a.shape, fill, dtype=float)
    m = np.abs(b) > 1e-12
    out[m] = a[m] / b[m]
    return out


def _wrap(a):
    """Wrap angles to (-pi, pi]."""
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


def _ffill_invalid(values, valid):
    """Replace invalid entries with the last valid one (vectorised)."""
    values = np.asarray(values, dtype=float)
    valid = np.asarray(valid, dtype=bool)
    if not valid.any():
        return np.zeros_like(values)
    idx = np.where(valid, np.arange(len(values)), 0)
    idx = np.maximum.accumulate(idx)
    return values[idx]


def _stats(prefix, arr, pcts=(25, 50, 75)):
    """mean/std/min/max/percentiles of an array, as a flat dict."""
    out = {}
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        keys = ["mean", "std", "min", "max"] + [f"p{p}" for p in pcts]
        return {f"{prefix}_{k}": np.nan for k in keys}
    out[f"{prefix}_mean"] = float(arr.mean())
    out[f"{prefix}_std"] = float(arr.std(ddof=0))
    out[f"{prefix}_min"] = float(arr.min())
    out[f"{prefix}_max"] = float(arr.max())
    for p, q in zip(pcts, np.percentile(arr, pcts)):
        out[f"{prefix}_p{p}"] = float(q)
    return out


# ----------------------------------------------------------------------------
# 1. load
# ----------------------------------------------------------------------------


def load_events(path, cfg: Config = Config()) -> pd.DataFrame:
    """Read a raw log and return a clean frame with columns t, x, y, state, button, is_move."""
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]

    tcol = cfg.time_col.strip().lower()
    if tcol not in df.columns:
        raise KeyError(f"time column {tcol!r} not in {list(df.columns)}")
    for c in ("x", "y", "state"):
        if c not in df.columns:
            raise KeyError(f"expected column {c!r} in {list(df.columns)}")

    out = pd.DataFrame(
        {
            "t": pd.to_numeric(df[tcol], errors="coerce"),
            "x": pd.to_numeric(df["x"], errors="coerce"),
            "y": pd.to_numeric(df["y"], errors="coerce"),
            "state": df["state"].astype(str).str.strip().str.lower(),
            "button": df.get("button", "NoButton").astype(str).str.strip().str.lower(),
        }
    ).dropna(subset=["t", "x", "y"])

    # Balabit sometimes parks the cursor at absurd coordinates on session start/end
    out = out[(out.x.between(-1e4, 1e5)) & (out.y.between(-1e4, 1e5))]

    out = out.sort_values("t", kind="mergesort").reset_index(drop=True)
    out["is_move"] = out.state.isin(MOVE_STATES)

    # Ties: several move samples share one timestamp (event coalescing in the log).
    # Keep the LAST position of each tied group; never drop button events.
    moves = out[out.is_move].drop_duplicates(subset="t", keep="last")
    others = out[~out.is_move]
    out = (
        pd.concat([moves, others])
        .sort_values(["t"], kind="mergesort")
        .reset_index(drop=True)
    )

    if cfg.screen_w:
        out["x"] = out["x"] / cfg.screen_w
    if cfg.screen_h:
        out["y"] = out["y"] / cfg.screen_h

    return out


def diagnose(events: pd.DataFrame, cfg: Config = Config()) -> dict:
    """Sampling-rate report. Run this once per dataset before trusting any derivative."""
    mv = events[events.is_move]
    dt = np.diff(mv.t.values)
    dt = dt[dt > 0]
    d = {
        "n_events": len(events),
        "n_moves": len(mv),
        "n_clicks": int(events.state.isin(DOWN_STATES).sum()),
        "duration_s": float(events.t.max() - events.t.min()) if len(events) else 0.0,
        "dt_median": float(np.median(dt)) if dt.size else np.nan,
        "dt_p90": float(np.percentile(dt, 90)) if dt.size else np.nan,
        "sample_rate_hz": float(1.0 / np.median(dt)) if dt.size else np.nan,
    }
    if cfg.verbose:
        print(
            f"[diagnose] {d['n_moves']} moves, {d['n_clicks']} clicks, "
            f"{d['duration_s']:.0f}s, median dt={d['dt_median']*1000:.0f}ms "
            f"(~{d['sample_rate_hz']:.0f} Hz)"
        )
        if d["sample_rate_hz"] < 30:
            print(
                "  ! low sampling rate: jerk and spectral features will be mostly "
                "quantisation noise. Trust speed/geometry/timing features instead."
            )
        if cfg.pause_split_s < 3 * d["dt_median"]:
            print(
                f"  ! pause_split_s={cfg.pause_split_s}s is close to the sampling "
                f"interval; almost every sample would start a new stroke."
            )
    return d


# ----------------------------------------------------------------------------
# 2. click pairing
# ----------------------------------------------------------------------------


def pair_clicks(events: pd.DataFrame) -> pd.DataFrame:
    """Match each press to the next release of the same button -> dwell time."""
    ev = events[events.state.isin(DOWN_STATES | UP_STATES)]
    rows, open_press = [], {}
    for r in ev.itertuples():
        if r.state in DOWN_STATES:
            open_press[r.button] = r
        elif r.state in UP_STATES and r.button in open_press:
            p = open_press.pop(r.button)
            rows.append(
                {
                    "t_down": p.t,
                    "t_up": r.t,
                    "dwell": r.t - p.t,
                    "button": r.button,
                    "cx": p.x,
                    "cy": p.y,
                }
            )
    return pd.DataFrame(rows, columns=["t_down", "t_up", "dwell", "button", "cx", "cy"])


# ----------------------------------------------------------------------------
# 3. segmentation
# ----------------------------------------------------------------------------


def segment_strokes(events: pd.DataFrame, cfg: Config = Config()):
    """Cut the move stream into strokes. Returns a list of (t, x, y, is_drag, click_row_or_None)."""
    mv = events[events.is_move].reset_index(drop=True)
    if len(mv) < 2:
        return []

    thr = cfg.pause_split_s
    if cfg.adaptive_pause:
        dt_all = np.diff(mv.t.values)
        dt_all = dt_all[dt_all > 0]
        if dt_all.size:
            thr = max(thr, cfg.adaptive_pause_mult * float(np.median(dt_all)))

    t = mv.t.values.astype(float)
    x = mv.x.values.astype(float)
    y = mv.y.values.astype(float)
    is_drag = (mv.state == "drag").values

    # how many click events have happened before each move sample
    click_cum = events.state.isin(DOWN_STATES | UP_STATES).cumsum()
    click_cum_mv = click_cum[events.is_move.values].values

    dt = np.diff(t)
    brk = np.zeros(len(t), dtype=bool)
    brk[1:] |= dt > thr                       # pause
    brk[1:] |= dt <= 0                        # clock glitch
    brk[1:] |= is_drag[1:] != is_drag[:-1]    # drag <-> free move
    if cfg.split_on_click:
        brk[1:] |= np.diff(click_cum_mv) > 0  # a click happened in between

    sid = np.cumsum(brk)

    clicks = pair_clicks(events)
    strokes = []
    for _, idx in pd.Series(np.arange(len(t))).groupby(sid):
        idx = idx.values
        # hard caps: chop over-long strokes into pieces
        pieces = [idx]
        if len(idx) > cfg.max_points:
            pieces = [
                idx[i : i + cfg.max_points] for i in range(0, len(idx), cfg.max_points)
            ]
        for p in pieces:
            if len(p) < 2:
                continue
            tt, xx, yy = t[p], x[p], y[p]
            if tt[-1] - tt[0] > cfg.max_duration_s:
                keep = tt - tt[0] <= cfg.max_duration_s
                tt, xx, yy = tt[keep], xx[keep], yy[keep]
                if len(tt) < 2:
                    continue
            # click that terminates this stroke, if any (press within 1s after the end)
            click = None
            if len(clicks):
                cand = clicks[(clicks.t_down >= tt[-1] - 1e-9) & (clicks.t_down <= tt[-1] + 1.0)]
                if len(cand):
                    click = cand.iloc[0]
            strokes.append((tt, xx, yy, bool(is_drag[p[0]]), click))
    return strokes


# ----------------------------------------------------------------------------
# 4. per-stroke features
# ----------------------------------------------------------------------------


def stroke_features(t, x, y, is_drag=False, click=None, cfg: Config = Config()) -> dict:
    n = len(t)
    f: dict = {}

    dt = np.diff(t)
    dx, dy = np.diff(x), np.diff(y)
    seg = np.hypot(dx, dy)
    v = _safe_div(seg, dt)

    duration = float(t[-1] - t[0])
    path = float(seg.sum())
    disp = float(np.hypot(x[-1] - x[0], y[-1] - y[0]))

    # --- shape ---------------------------------------------------------------
    f["n_points"] = n
    f["duration"] = duration
    f["path_len"] = path
    f["displacement"] = disp
    f["straightness"] = disp / path if path > 0 else 0.0
    f["is_drag"] = int(is_drag)

    bw, bh = float(x.max() - x.min()), float(y.max() - y.min())
    f["bbox_w"], f["bbox_h"] = bw, bh
    f["bbox_area"] = bw * bh
    f["bbox_aspect"] = np.log1p(bw) - np.log1p(bh)          # symmetric, no div-by-zero
    f["angle_start_end"] = float(np.arctan2(y[-1] - y[0], x[-1] - x[0]))
    f["dir_sin"] = float(np.sin(f["angle_start_end"]))      # layout-invariant direction
    f["dir_cos"] = float(np.cos(f["angle_start_end"]))

    # deviation from the straight start->end line ("how bowed is the path")
    if disp > 1e-9:
        ux, uy = (x[-1] - x[0]) / disp, (y[-1] - y[0]) / disp
        perp = np.abs((x - x[0]) * (-uy) + (y - y[0]) * ux)
        f["dev_mean"] = float(perp.mean())
        f["dev_max"] = float(perp.max())
        f["dev_max_norm"] = float(perp.max() / disp)
    else:
        f["dev_mean"] = f["dev_max"] = f["dev_max_norm"] = 0.0

    # --- speed ---------------------------------------------------------------
    f.update(_stats("v", v))
    f["v_cv"] = f["v_std"] / f["v_mean"] if f["v_mean"] else np.nan
    tm = 0.5 * (t[1:] + t[:-1])                              # midpoint times for v
    if v.size:
        f["t_peak_frac"] = float((tm[int(np.argmax(v))] - t[0]) / duration) if duration > 0 else np.nan
        f["v_terminal"] = float(v[-1])
        f["v_initial"] = float(v[0])
    else:
        f["t_peak_frac"] = f["v_terminal"] = f["v_initial"] = np.nan

    # --- acceleration / jerk (weak at low sample rates -- see diagnose()) -----
    if n >= 3:
        a = _safe_div(np.diff(v), np.diff(tm))
        f.update(_stats("a", a))
        f["a_abs_mean"] = float(np.abs(a).mean())
        f["accel_frac"] = float((a > 0).mean())              # share of time speeding up
    else:
        f.update(_stats("a", np.array([])))
        f["a_abs_mean"] = f["accel_frac"] = np.nan

    if n >= 4:
        tj = 0.5 * (tm[1:] + tm[:-1])
        j = _safe_div(np.diff(a), np.diff(tj))
        f["j_abs_mean"] = float(np.abs(j).mean())
        f["j_abs_max"] = float(np.abs(j).max())
        f["j_std"] = float(j.std(ddof=0))
    else:
        f["j_abs_mean"] = f["j_abs_max"] = f["j_std"] = np.nan

    # --- angular ------------------------------------------------------------
    th = _ffill_invalid(np.arctan2(dy, dx), seg > 1e-12)
    if th.size >= 2:
        dth = _wrap(np.diff(th))
        angv = _safe_div(dth, np.diff(tm))
        curv = _safe_div(dth, 0.5 * (seg[1:] + seg[:-1]))
        f["angle_abs_sum"] = float(np.abs(dth).sum())
        f["angle_abs_mean"] = float(np.abs(dth).mean())
        f["angv_abs_mean"] = float(np.abs(angv).mean())
        f["angv_abs_max"] = float(np.abs(angv).max())
        f["curv_abs_mean"] = float(np.abs(curv).mean())
        f["curv_abs_max"] = float(np.abs(curv).max())
        f["curv_std"] = float(curv.std(ddof=0))
        nd = int((np.abs(dth) > np.deg2rad(cfg.dir_change_deg)).sum())
        f["n_dir_changes"] = nd
        f["dir_change_rate"] = nd / duration if duration > 0 else np.nan
    else:
        for k in (
            "angle_abs_sum angle_abs_mean angv_abs_mean angv_abs_max "
            "curv_abs_mean curv_abs_max curv_std n_dir_changes dir_change_rate"
        ).split():
            f[k] = np.nan

    # --- sampling / rhythm ---------------------------------------------------
    f["dt_mean"] = float(dt.mean())
    f["dt_std"] = float(dt.std(ddof=0))
    f["dt_cv"] = f["dt_std"] / f["dt_mean"] if f["dt_mean"] else np.nan
    micro = seg < cfg.stationary_px
    f["n_micro_pauses"] = int(micro.sum())
    f["micro_pause_frac"] = float(dt[micro].sum() / duration) if duration > 0 else np.nan

    # --- final approach (the part that discriminates most, per the literature) -
    tail = tm >= (t[-1] - cfg.tail_frac * duration) if duration > 0 else np.zeros_like(tm, bool)
    if tail.any():
        f["tail_v_mean"] = float(v[tail].mean())
        f["tail_v_max"] = float(v[tail].max())
        f["tail_path_frac"] = float(seg[tail].sum() / path) if path > 0 else np.nan
    else:
        f["tail_v_mean"] = f["tail_v_max"] = f["tail_path_frac"] = np.nan

    # --- click ---------------------------------------------------------------
    if click is not None:
        cx, cy = float(click.cx), float(click.cy)
        d = np.hypot(x - cx, y - cy)
        i = int(np.argmin(d))
        f["has_click"] = 1
        f["click_dwell"] = float(click.dwell)
        f["click_button_left"] = int(str(click.button).startswith("left"))
        f["click_delay"] = float(click.t_down - t[-1])       # move end -> press
        f["click_dist_end"] = float(np.hypot(x[-1] - cx, y[-1] - cy))
        f["overshoot_path"] = float(seg[i:].sum())           # path travelled after closest approach
        f["overshoot_max"] = float(d[i:].max())
    else:
        for k in (
            "has_click click_dwell click_button_left click_delay "
            "click_dist_end overshoot_path overshoot_max"
        ).split():
            f[k] = 0 if k == "has_click" else np.nan

    return f


# ----------------------------------------------------------------------------
# 5. file / directory drivers
# ----------------------------------------------------------------------------


def strokes_to_frame(strokes, cfg: Config = Config()) -> pd.DataFrame:
    rows, dropped = [], {"short": 0, "tiny_path": 0, "brief": 0}
    for k, (t, x, y, is_drag, click) in enumerate(strokes):
        if len(t) < cfg.min_points:
            dropped["short"] += 1
            continue
        if float(np.hypot(np.diff(x), np.diff(y)).sum()) < cfg.min_path_px:
            dropped["tiny_path"] += 1
            continue
        if t[-1] - t[0] < cfg.min_duration_s:
            dropped["brief"] += 1
            continue
        f = stroke_features(t, x, y, is_drag, click, cfg)
        f["stroke_id"] = k
        f["t_start"] = float(t[0])
        f["t_end"] = float(t[-1])
        rows.append(f)

    if cfg.verbose:
        print(
            f"[strokes] kept {len(rows)} / {len(strokes)} "
            f"(dropped: {dropped['short']} too few points, "
            f"{dropped['tiny_path']} too short, {dropped['brief']} too brief)"
        )
    df = pd.DataFrame(rows)
    return df.replace([np.inf, -np.inf], np.nan)


def process_file(path, cfg: Config = Config(), user=None, session=None) -> pd.DataFrame:
    ev = load_events(path, cfg)
    if cfg.verbose:
        diagnose(ev, cfg)
    df = strokes_to_frame(segment_strokes(ev, cfg), cfg)
    if len(df):
        df.insert(0, "session", session or Path(path).name)
        df.insert(0, "user", user or Path(path).parent.name)
    return df


def process_dir(root, cfg: Config = Config(), pattern="**/session_*") -> pd.DataFrame:
    """Walk a Balabit-style tree (root/userN/session_xxxx) into one long frame."""
    root = Path(root)
    out = []
    for p in sorted(root.glob(pattern)):
        if not p.is_file():
            continue
        try:
            d = process_file(p, cfg, user=p.parent.name, session=p.name)
            if len(d):
                out.append(d)
        except Exception as e:  # keep going -- some session files are truncated
            warnings.warn(f"{p}: {e}")
    if not out:
        return pd.DataFrame()
    df = pd.concat(out, ignore_index=True)
    if cfg.verbose:
        print(f"[process_dir] {len(df)} strokes from {df.user.nunique()} users")
    return df


# ----------------------------------------------------------------------------
# 6. optional: strokes -> model input windows
# ----------------------------------------------------------------------------


NON_FEATURE = {"user", "session", "stroke_id", "t_start", "t_end"}


def strokes_to_windows(
    df: pd.DataFrame,
    n: int = 25,
    stride: int = 5,
    min_strokes: int = 8,
    max_span_s: float = 60.0,
    aggs=("mean", "std", "p25", "p50", "p75"),
) -> pd.DataFrame:
    """Sliding window over strokes -> one wide vector per window.

    Windows spanning more than max_span_s are dropped (the user walked away).
    Windows with fewer than min_strokes are never emitted (idle reading).
    """
    feat_cols = [c for c in df.columns if c not in NON_FEATURE]
    rows = []
    for (u, s), g in df.groupby(["user", "session"], sort=False):
        g = g.sort_values("t_start").reset_index(drop=True)
        vals = g[feat_cols].to_numpy(dtype=float)
        for end in range(n, len(g) + 1, stride):
            start = end - n
            if end - start < min_strokes:
                continue
            span = g.t_end.iloc[end - 1] - g.t_start.iloc[start]
            if span > max_span_s:
                continue
            blk = vals[start:end]
            rec = {"user": u, "session": s,
                   "t_start": float(g.t_start.iloc[start]),
                   "t_end": float(g.t_end.iloc[end - 1]),
                   "n_strokes": end - start, "span_s": float(span)}
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                for agg in aggs:
                    if agg == "mean":
                        vv = np.nanmean(blk, axis=0)
                    elif agg == "std":
                        vv = np.nanstd(blk, axis=0)
                    else:
                        vv = np.nanpercentile(blk, int(agg[1:]), axis=0)
                    for c, val in zip(feat_cols, vv):
                        rec[f"{c}_{agg}"] = float(val)
            rows.append(rec)
    out = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    print(f"[windows] {len(out)} windows x {len([c for c in out.columns if c not in NON_FEATURE | {'n_strokes','span_s'}])} features")
    return out


# ----------------------------------------------------------------------------


In [6]:
from dataclasses import dataclass, asdict, replace
import numpy as np, pandas as pd, warnings, json, time

@dataclass
class PrepConfig:
    # --- event level ---
    x_range: tuple = (-2000.0, 8000.0)
    y_range: tuple = (-2000.0, 5000.0)
    drop_scroll: bool = True
    drop_out_of_order: bool = True
    despike: bool = True
    despike_px: float = 250.0
    despike_ratio: float = 0.35
    collapse_frozen: bool = False
    frozen_px: float = 0.0
    # --- session level ---
    min_events: int = 150
    min_moves: int = 100
    min_duration_s: float = 20.0
    max_dt_median_s: float = 0.40
    # --- stroke level ---
    max_speed_px_s: float = 6000.0
    max_path_px: float = 20000.0
    drop_zero_var_strokes: bool = True
    # --- feature level ---
    max_nan_frac: float = 0.50
    min_unique: int = 3
    winsor_mad: float = 6.0
    log1p_skew: float = 3.0
    scaler: str = "robust"          # "robust" | "standard" | "none"
    corr_prune: float = 0.0         # 0 disables; else drop |r| above this


def _despike_mask(x, y, jump_px, ratio):
    n = len(x)
    bad = np.zeros(n, dtype=bool)
    if n < 3:
        return bad
    d = np.hypot(np.diff(x), np.diff(y))
    a, b = d[:-1], d[1:]
    c = np.hypot(x[2:] - x[:-2], y[2:] - y[:-2])
    bad[1:-1] = (a > jump_px) & (b > jump_px) & (c < ratio * (a + b))
    return bad


def clean_events(raw: pd.DataFrame, cfg: Config = Config(), pcfg: PrepConfig = PrepConfig()):
    rep = {"n_raw": len(raw)}
    df = raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    tcol = cfg.time_col.strip().lower()
    for c in (tcol, "x", "y", "state"):
        if c not in df.columns:
            raise KeyError(f"expected column {c!r} in {list(df.columns)}")

    out = pd.DataFrame({
        "t":      pd.to_numeric(df[tcol], errors="coerce").astype("float64"),
        "x":      pd.to_numeric(df["x"], errors="coerce").astype("float64"),
        "y":      pd.to_numeric(df["y"], errors="coerce").astype("float64"),
        "state":  df["state"].astype(str).str.strip().str.lower(),
        "button": (df["button"] if "button" in df.columns else "nobutton").astype(str).str.strip().str.lower(),
    })

    n = len(out); out = out.dropna(subset=["t", "x", "y"]);            rep["drop_nan"] = n - len(out)

    if pcfg.drop_scroll:
        n = len(out)
        out = out[~(out.button.str.contains("scroll") | out.state.str.contains("scroll"))]
        rep["drop_scroll"] = n - len(out)

    n = len(out)
    out = out[out.x.between(*pcfg.x_range) & out.y.between(*pcfg.y_range)];  rep["drop_oob"] = n - len(out)

    if pcfg.drop_out_of_order and len(out):
        t = out.t.values
        keep = t >= np.maximum.accumulate(t) - 1e-9
        rep["drop_backwards"] = int((~keep).sum())
        out = out[keep]
    else:
        rep["drop_backwards"] = 0

    out = out.sort_values("t", kind="mergesort").reset_index(drop=True)
    out["is_move"] = out.state.isin(MOVE_STATES)

    n = len(out)
    moves  = out[out.is_move].drop_duplicates(subset="t", keep="last")
    others = out[~out.is_move]
    out = pd.concat([moves, others]).sort_values("t", kind="mergesort").reset_index(drop=True)
    rep["drop_tied_t"] = n - len(out)

    if pcfg.despike and out.is_move.any():
        mv = out.index[out.is_move.values].to_numpy()
        bad = _despike_mask(out.x.values[mv], out.y.values[mv], pcfg.despike_px, pcfg.despike_ratio)
        rep["drop_spikes"] = int(bad.sum())
        if bad.any():
            out = out.drop(index=mv[bad]).reset_index(drop=True)
    else:
        rep["drop_spikes"] = 0

    if pcfg.collapse_frozen and out.is_move.any():
        mv = out.is_move.values
        same = np.zeros(len(out), dtype=bool)
        same[1:] = mv[1:] & mv[:-1] & (np.abs(np.diff(out.x.values)) <= pcfg.frozen_px) \
                                    & (np.abs(np.diff(out.y.values)) <= pcfg.frozen_px)
        rep["drop_frozen"] = int(same.sum())
        out = out[~same].reset_index(drop=True)
    else:
        rep["drop_frozen"] = 0

    if cfg.screen_w: out["x"] = out["x"] / cfg.screen_w
    if cfg.screen_h: out["y"] = out["y"] / cfg.screen_h

    rep["n_clean"] = len(out)
    return out.reset_index(drop=True), rep


def session_report(ev: pd.DataFrame, pcfg: PrepConfig = PrepConfig()) -> dict:
    mv = ev[ev.is_move]
    dt = np.diff(mv.t.values); dt = dt[dt > 0]
    d = {
        "n_events":  len(ev),
        "n_moves":   len(mv),
        "n_clicks":  int(ev.state.isin(DOWN_STATES).sum()),
        "duration_s": float(ev.t.max() - ev.t.min()) if len(ev) else 0.0,
        "dt_median": float(np.median(dt)) if dt.size else np.nan,
        "hz":        float(1 / np.median(dt)) if dt.size else np.nan,
    }
    reasons = []
    if d["n_events"]   < pcfg.min_events:      reasons.append("few_events")
    if d["n_moves"]    < pcfg.min_moves:       reasons.append("few_moves")
    if d["duration_s"] < pcfg.min_duration_s:  reasons.append("short")
    if not np.isfinite(d["dt_median"]) or d["dt_median"] > pcfg.max_dt_median_s:
        reasons.append("slow_sampling")
    d["keep"] = len(reasons) == 0
    d["drop_reason"] = ",".join(reasons)
    return d

In [7]:
PREP = PrepConfig()

def load_events(path, cfg: Config = Config()) -> pd.DataFrame:      # noqa: F811
    """Overrides the version in the feature cell so file- and memory-paths clean identically."""
    return clean_events(pd.read_csv(path), cfg, PREP)[0]


def process_frame(raw: pd.DataFrame, user: str, session: str,
                  cfg: Config = Config(), pcfg: PrepConfig = PREP, keep_seqs: bool = True):
    """Returns (stroke feature table, QC record, raw point sequences).

    The feature table is no longer model input -- the encoder eats the raw
    sequences. It is kept because every QC threshold downstream (`v_max`,
    `path_len`, `duration`) is defined on it, and because it is the baseline the
    raw model has to beat.
    """
    ev, rep = clean_events(raw, cfg, pcfg)
    qc = {"user": user, "session": session, **rep, **session_report(ev, pcfg)}
    if not qc["keep"]:
        return pd.DataFrame(), qc, {}
    raw_strokes = segment_strokes(ev, cfg)
    st = strokes_to_frame(raw_strokes, cfg)
    qc["n_strokes"] = len(st)
    seqs = {}
    if len(st):
        st.insert(0, "session", session)
        st.insert(0, "user", user)
        if keep_seqs:
            # stroke_id indexes raw_strokes, so surviving strokes map straight back
            for sid in st.stroke_id.to_numpy():
                t, x, y, is_drag, _click = raw_strokes[int(sid)]
                seqs[(user, session, int(sid))] = (
                    np.asarray(t, dtype=np.float64),
                    np.asarray(x, dtype=np.float32),
                    np.asarray(y, dtype=np.float32),
                    bool(is_drag),
                )
    else:
        qc["keep"], qc["drop_reason"] = False, "no_strokes"
    return st, qc, seqs


def process_loaded(users: dict, cfg: Config = Config(), pcfg: PrepConfig = PREP,
                   progress_every: int = 100, keep_seqs: bool = True):
    cfg = replace(cfg, verbose=False)
    frames, qcs, seqs, t0, i = [], [], {}, time.time(), 0
    total = sum(len(s) for s in users.values())
    for u, sessions in users.items():
        for sname, raw in sessions.items():
            i += 1
            try:
                st, qc, sq = process_frame(raw, u, sname, cfg, pcfg, keep_seqs)
            except Exception as e:
                qcs.append({"user": u, "session": sname, "keep": False,
                            "drop_reason": f"error:{type(e).__name__}"})
                warnings.warn(f"{u}/{sname}: {e}")
                continue
            qcs.append(qc)
            if len(st):
                frames.append(st); seqs.update(sq)
            if progress_every and i % progress_every == 0:
                print(f"  {i}/{total} sessions  ({time.time()-t0:.0f}s)")
    qc = pd.DataFrame(qcs)
    strokes = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    n_pts = sum(len(v[0]) for v in seqs.values())
    print(f"[pipeline] {qc.keep.sum()}/{len(qc)} sessions kept, "
          f"{len(strokes):,} strokes, {n_pts:,} raw points, {time.time()-t0:.0f}s")
    if (~qc.keep).any():
        print(qc.loc[~qc.keep, "drop_reason"].value_counts().to_string())
    return strokes, qc, seqs


# --- run the pipeline -------------------------------------------------------
# 59 Hz capture, so the defaults tuned for Balabit's ~9 Hz need revisiting:
# min_points=6 is 0.1 s here rather than 0.7 s, and max_points=300 is 5 s.
cfg = Config(time_col="client timestamp", verbose=False,
             pause_split_s=0.30,      # shorter: at 59 Hz a 0.5 s gap is 30 samples
             min_points=10,           # ~0.17 s -- below this a "stroke" is noise
             max_points=400)
# max_speed_px_s=6000 is a Balabit number and it is wrong here. At ~9 Hz,
# consecutive samples are 110 ms apart and instantaneous speed is smoothed over
# that interval; at 17 ms the true peaks are visible. Measured on this corpus the
# median stroke peaks at 4,670 px/s, so 6,000 sits at the 68th percentile of
# ORDINARY strokes and discards 34% of them -- with a per-user drop rate ranging
# 2.5%-71%, which is this notebook's own signature of a filter eating signal
# rather than artifacts. 20,000 drops 0.3% and still catches real teleports.
PREP = PrepConfig(min_events=100, min_moves=80, min_duration_s=20.0,
                  max_dt_median_s=0.10,    # reject anything not actually high-rate
                  max_speed_px_s=20000.0)

strokes_raw, qc, SEQS = process_loaded(users_raw, cfg, PREP, progress_every=40)
display(qc.head())
print("\nsampling rate across sessions (Hz):")
print(qc.hz.describe(percentiles=[.05, .5, .95]).round(1).to_string())

  40/245 sessions  (2s)
  80/245 sessions  (4s)
  120/245 sessions  (6s)
  160/245 sessions  (8s)
  200/245 sessions  (9s)
  240/245 sessions  (12s)
[pipeline] 245/245 sessions kept, 29,518 strokes, 1,073,547 raw points, 12s

sampling rate across sessions (Hz):
count    245.0
mean      65.3
std       21.8
min       32.3
5%        58.8
50%       58.8
95%      125.0
max      142.9
    user                         session  n_raw  drop_nan  drop_scroll  drop_oob  drop_backwards  drop_tied_t  drop_spikes  drop_frozen  n_clean  n_events  n_moves  n_clicks  duration_s  dt_median         hz  keep drop_reason  n_strokes
0  user1  user1__session_2020_05_14_1min   2055         0            0         0               0            0            0            0     2055      2055     1953        51      57.345      0.017  58.823529  True                     46
1  user1  user1__session_2020_05_14_3min   6293         0            0         0               0            0            0            0     6293

## 3. Stroke-level QC

Drops individual strokes that survived segmentation but violate physics —
`max_speed_px_s = 6000` is the operative one. Keep the thresholds loose: a user
who flings the mouse is *supposed* to look extreme, and over-filtering removes the
between-user variance the model is trying to find.

In [9]:
def filter_strokes(df: pd.DataFrame, pcfg: PrepConfig = PREP) -> pd.DataFrame:
    n0 = len(df); drops = {}
    m = pd.Series(True, index=df.index)

    bad = df.v_max > pcfg.max_speed_px_s;        drops["impossible_speed"] = int(bad.sum()); m &= ~bad
    bad = df.path_len > pcfg.max_path_px;        drops["huge_path"] = int(bad.sum());        m &= ~bad
    if pcfg.drop_zero_var_strokes:
        bad = (df.v_std == 0) | (df.bbox_area == 0)
        drops["degenerate"] = int(bad.sum()); m &= ~bad
    bad = ~np.isfinite(df.duration) | (df.duration <= 0)
    drops["bad_duration"] = int(bad.sum()); m &= ~bad

    out = df[m].reset_index(drop=True)
    print(f"[strokes] {len(out):,}/{n0:,} kept  " +
          "  ".join(f"-{k}:{v}" for k, v in drops.items() if v))
    return out


strokes = filter_strokes(strokes_raw, PREP)
display(pd.DataFrame({'raw': strokes_raw.groupby('user').size(),
                      'kept': strokes.groupby('user').size()}).describe().round(1))

[strokes] 29,413/29,518 kept  -impossible_speed:76  -degenerate:29
         raw   kept
count  120.0  120.0
mean   246.0  245.1
std     69.4   69.5
min    136.0  136.0
25%    203.8  202.8
50%    240.0  239.0
75%    274.0  274.0
max    599.0  599.0


## 4. Protocol — user-disjoint, enrol/probe by session

This is the part Balabit could not support. With 10 users, holding any out cost
too much training signal, so every number was closed-set: the model had already
seen every identity it was later asked about. That measures memorisation as much
as generalisation.

With 120 users the honest protocol is affordable:

| fold | users | role |
|---|---|---|
| `train` | 75 | encoder training, all sessions |
| `val_enrol` / `val_probe` | 15 | early stopping, **never trained on** |
| `test_enrol` / `test_probe` | 30 | the reported result, **never trained on** |

Inside every held-out user, enrolment uses the longest session and probing the
rest, so enrol and probe are always different recordings — within-session
similarity cannot be exploited.

Two leaks are asserted against in `role_index`: no held-out user appears in
training, and no session appears in both enrol and probe.

In [11]:
# ---------------------------------------------------------------------------
# Protocol: user-disjoint folds, enrol on one session and probe on another
# ---------------------------------------------------------------------------


def assign_user_folds(strokes: pd.DataFrame, n_val_users=15, n_test_users=30, seed=0):
    """Split by USER, then inside each held-out user split by SESSION.

    Balabit could not support this: with 10 users, holding any out cost too much
    training signal, so every number was closed-set -- the model had seen every
    identity it was later asked about. With 120 users the honest protocol becomes
    affordable, and it is the one that matches deployment: a person the encoder
    has never trained on is enrolled from one recording and verified from another.

    Enrolment uses the longest session and probing the rest, so enrol/probe are
    always different recordings and within-session similarity cannot be exploited.
    """
    rng = np.random.default_rng(seed)
    us = np.array(sorted(strokes.user.unique()))
    rng.shuffle(us)
    test_u = set(us[:n_test_users])
    val_u = set(us[n_test_users:n_test_users + n_val_users])

    out = strokes.copy()
    role = pd.Series("train", index=out.index, dtype=object)
    demoted = 0
    for u, g in out.groupby("user", sort=False):
        if u not in test_u and u not in val_u:
            continue
        if g.session.nunique() < 2:
            demoted += 1                    # cannot form an enrol/probe pair
            continue
        span = g.groupby("session").t_end.max() - g.groupby("session").t_start.min()
        enrol = span.idxmax()
        pre = "test" if u in test_u else "val"
        role[g.index] = np.where(g.session.to_numpy() == enrol,
                                 f"{pre}_enrol", f"{pre}_probe")
    out["fold"] = role
    if demoted:
        print(f"  ! {demoted} held-out users had one session only -> left in train")
    tab = out.groupby("fold").agg(strokes=("user", "size"), sessions=("session", "nunique"),
                                  users=("user", "nunique"))
    print(tab.to_string())
    return out


def build_window_index(df: pd.DataFrame, mcfg):
    """Sliding window over each session -> (M, n_strokes) row indices into the bank."""
    df = df.reset_index(drop=True)
    rows, y, sess, folds, spans, times = [], [], [], [], [], []
    for (u, s), g in df.groupby(["user", "session"], sort=False):
        g = g.sort_values("t_start")
        pos = g.index.to_numpy()
        t0, t1 = g.t_start.to_numpy(), g.t_end.to_numpy()
        for end in range(mcfg.n_strokes, len(g) + 1, mcfg.stride):
            st = end - mcfg.n_strokes
            span = t1[end - 1] - t0[st]
            if span > mcfg.max_span_s:
                continue
            rows.append(pos[st:end]); y.append(u); sess.append(s)
            spans.append(span); times.append((t0[st], t1[end - 1]))
            folds.append(g["fold"].iloc[0])
    if not rows:
        raise RuntimeError("no windows -- lower n_strokes or raise max_span_s")
    W = np.stack(rows).astype(np.int64)
    y, sess, folds = np.array(y), np.array(sess), np.array(folds)
    wmeta = np.asarray(times, dtype=np.float64)
    print(f"[windows] {len(W):,} x {W.shape[1]} strokes | {len(np.unique(y))} users "
          f"| median span {np.median(spans):.1f}s")
    cnt = pd.Series(y).value_counts()
    print(f"[windows] per-user: min {cnt.min()}, median {int(cnt.median())}, max {cnt.max()}")
    return W, y, sess, folds, wmeta


def role_index(folds, sess, y):
    out = {f: np.where(folds == f)[0] for f in
           ("train", "val_enrol", "val_probe", "test_enrol", "test_probe")
           if (folds == f).any()}
    print("[roles] " + " | ".join(
        f"{k}: {len(v):,} win / {len(set(y[v]))} users" for k, v in out.items()))
    tr_u = set(y[out["train"]])
    for k in ("val_enrol", "val_probe", "test_enrol", "test_probe"):
        if k in out:
            assert not (tr_u & set(y[out[k]])), f"user leak: {k} overlaps train"
    for a, b in (("val_enrol", "val_probe"), ("test_enrol", "test_probe")):
        if a in out and b in out:
            assert not (set(sess[out[a]]) & set(sess[out[b]])), f"session leak {a}/{b}"
    print("[roles] no user leak into train, no session leak between enrol and probe")
    return out

## 5. Model

Two levels. A stroke becomes a vector — by the feature table or by the CNN,
depending on the notebook — and then a permutation-invariant set encoder pools
`n_strokes` of them into one L2-normalised embedding. Stroke order inside a
window is close to arbitrary, and an order-sensitive encoder over it mostly
learns session-specific sequencing, which is exactly what fails to transfer.

The loss is **supervised contrastive**, not triplet. Both triplet forms collapsed
the Balabit model: they penalise the difference `dp - dn`, which scales with the
embedding radius, so shrinking everything toward one point reduces the loss
monotonically and collapse is a descent direction. `supcon` is a softmax over
cosine similarities, where a collapsed embedding makes every logit equal and the
loss hits its *maximum*. Watch `spread` in the training log regardless — it is the
canary, and it is printed every epoch.

In [13]:
NON_FEATURE_ALL = {"user", "session", "stroke_id", "t_start", "t_end", "n_strokes", "span_s"}
# structurally-missing columns: NaN means "no click happened", not "unknown"
STRUCTURAL_NAN = {"click_dwell", "click_button_left", "click_delay", "click_dist_end",
                  "overshoot_path", "overshoot_max"}


def fit_preprocessor(train: pd.DataFrame, pcfg: PrepConfig = PREP) -> dict:
    feats = [c for c in train.columns if c not in NON_FEATURE_ALL]
    X = train[feats].astype("float64")

    keep = [c for c in feats
            if X[c].isna().mean() <= pcfg.max_nan_frac
            and X[c].nunique(dropna=True) >= pcfg.min_unique]
    dropped_cols = sorted(set(feats) - set(keep))
    X = X[keep]

    # heavy right tails -> log1p (only strictly non-negative columns)
    logs = [c for c in keep
            if (X[c].dropna() >= 0).all() and abs(X[c].skew(skipna=True)) > pcfg.log1p_skew]
    X[logs] = np.log1p(X[logs])

    # robust bounds from median / MAD, percentile fallback when MAD collapses
    med = X.median()
    mad = (X - med).abs().median() * 1.4826
    lo = med - pcfg.winsor_mad * mad
    hi = med + pcfg.winsor_mad * mad
    flat = mad <= 1e-12
    if flat.any():
        lo[flat] = X.loc[:, flat].quantile(0.005)
        hi[flat] = X.loc[:, flat].quantile(0.995)
    X = X.clip(lo, hi, axis=1)

    fill = {c: (0.0 if c in STRUCTURAL_NAN else float(med[c])) for c in keep}

    if pcfg.scaler == "robust":
        center = X.median()
        q1, q3 = X.quantile(0.25), X.quantile(0.75)
        scale = (q3 - q1).replace(0, np.nan).fillna(X.std(ddof=0)).replace(0, 1.0)
    elif pcfg.scaler == "standard":
        center, scale = X.mean(), X.std(ddof=0).replace(0, 1.0)
    else:
        center = pd.Series(0.0, index=keep); scale = pd.Series(1.0, index=keep)

    pre = {"features": keep, "dropped_cols": dropped_cols, "log_cols": logs,
           "lo": lo.to_dict(), "hi": hi.to_dict(), "fill": fill,
           "center": center.to_dict(), "scale": scale.to_dict(),
           "config": asdict(pcfg)}

    if pcfg.corr_prune:
        Z = apply_preprocessor(train, pre)
        cm = Z.corr().abs().to_numpy(copy=True)
        np.fill_diagonal(cm, 0.0)
        drop, cols = set(), list(Z.columns)
        for i in range(len(cols)):
            if cols[i] in drop: continue
            for j in range(i + 1, len(cols)):
                if cm[i, j] > pcfg.corr_prune: drop.add(cols[j])
        pre["features"] = [c for c in keep if c not in drop]
        pre["corr_pruned"] = sorted(drop)
        print(f"[prep] corr>|{pcfg.corr_prune}| pruned {len(drop)} columns")

    print(f"[prep] {len(pre['features'])} features "
          f"(dropped {len(dropped_cols)} constant/empty, log1p on {len(logs)})")
    return pre


def apply_preprocessor(df: pd.DataFrame, pre: dict) -> pd.DataFrame:
    """Returns FEATURES ONLY, row order preserved -- no user/session/t_start columns.

    Keeping metadata out of the matrix is deliberate: `t_start` is a position in the
    session, and a model handed that column will happily learn it.
    """
    cols = pre["features"]
    X = df.reindex(columns=cols).astype("float64")
    logc = [c for c in pre["log_cols"] if c in cols]
    if logc:
        X[logc] = np.log1p(X[logc].clip(lower=-0.999999))
    X = X.clip(pd.Series(pre["lo"])[cols], pd.Series(pre["hi"])[cols], axis=1)
    X = X.fillna(pd.Series(pre["fill"])[cols])
    X = (X - pd.Series(pre["center"])[cols]) / pd.Series(pre["scale"])[cols]
    return X.replace([np.inf, -np.inf], 0.0).reset_index(drop=True)


In [14]:
# Colab already has torch; uncomment if the runtime is bare.
# !pip -q install torch

import json, math, time, warnings
import joblib                     # save_model in 8.5; the raw path must not need the feature cells
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"                # Apple silicon; set DEVICE="cpu" if an op is unsupported
else:
    DEVICE = "cpu"
NON_FEATURE_ALL = {"user", "session", "stroke_id", "t_start", "t_end",
                   "n_strokes", "span_s", "fold"}   # never features
print("device:", DEVICE)


@dataclass
class ModelConfig:
    # --- windowing ---
    n_strokes: int = 12            # strokes per window (the unit that gets embedded)
    stride: int = 4
    max_span_s: float = 240.0      # drop windows where the user walked away mid-way
    # 12/4/240, not 25/8/90: at ~9 Hz a 25-stroke window routinely spans minutes, so
    # max_span_s=90 was silently discarding ~75% of all windows.
    # --- stroke front end (raw sequence -> one vector) ---
    d_conv: int = 64               # width of the temporal conv stack
    kernel: int = 5
    dilations: tuple = (1, 2, 4)   # receptive field ~ 1 + 2*(k-1)*sum(dil) = 29 samples
    d_stroke: int = 128            # per-stroke vector handed to the set encoder
    # --- window encoder ---
    d_hidden: int = 256
    d_embed: int = 64
    n_layers: int = 2
    dropout: float = 0.15
    pool: str = "attn+meanstd"     # "mean" | "meanstd" | "attn+meanstd"
    # --- loss ---
    loss: str = "supcon"           # "supcon" | "batch_hard" | "contrastive"
    temperature: float = 0.1       # supcon only
    margin: float = 0.0            # batch_hard only
    # Both triplet forms collapsed this dataset. They penalise the DIFFERENCE
    # dp - dn, which scales with the embedding radius, so shrinking everything
    # toward a single point reduces the loss monotonically: margin=0.25 parked at
    # loss==0.25, margin=0 parked at softplus(0)=ln2=0.693. supcon is a softmax
    # over cosine similarities and has the opposite behaviour -- a collapsed
    # embedding makes every logit equal, which is its WORST case, not its best.
    # --- optimisation ---
    P_users: int = 8               # users per batch
    K_windows: int = 6             # windows per user per batch  (batch = P*K)
    lr: float = 5e-4               # 2e-3 collapsed the embedding during the warm-up ramp
    weight_decay: float = 1e-2
    epochs: int = 60
    steps_per_epoch: int = 60
    patience: int = 12
    eval_max_windows: int = 3000   # per-epoch validation subsample; full sets in evaluate()
    embed_bs: int = 64
    seed: int = 0


class WindowBank:
    """Stroke bank + window index -> padded (B, N, L, C) batches with a mask.

    Scaling happens here, at batch time, rather than in the stored array: the
    bank holds raw pixels and seconds (exact in float16 -- coordinates are
    integers), so re-fitting the scaler never means rebuilding the bank.
    """

    kind = "raw"

    def __init__(self, S, lens, W, y, sess, folds, scaler=None, device=None):
        self.S, self.lens, self.W = S, lens, W
        self.y, self.sess, self.folds = y, sess, folds
        self.scaler = self.preproc = scaler
        self.device = device or DEVICE
        self._cache = None

    def make_encoder(self, mcfg):
        return RawWindowEncoder(self.n_channels, mcfg)

    def __len__(self):
        return len(self.W)

    @property
    def n_channels(self):
        return self.S.shape[2]

    @property
    def max_len(self):
        return self.S.shape[1]

    def _sc(self):
        if self._cache is None and self.scaler is not None:
            d = self.device
            self._cache = (
                torch.as_tensor(self.scaler["center"], dtype=torch.float32, device=d),
                torch.as_tensor(1.0 / np.asarray(self.scaler["scale"], np.float32),
                                dtype=torch.float32, device=d),
                torch.as_tensor(np.asarray(self.scaler["log"]), dtype=torch.bool, device=d),
            )
        return self._cache

    def batch(self, rows):
        idx = self.W[rows]                                          # (b, N)
        x = torch.from_numpy(np.ascontiguousarray(self.S[idx], dtype=np.float32)).to(self.device)
        n = torch.from_numpy(self.lens[idx].astype(np.int64)).to(self.device)
        ar = torch.arange(self.max_len, device=self.device)
        m = ar[None, None, :] < n[..., None]                        # (b, N, L)
        sc = self._sc()
        if sc is not None:
            center, inv, logm = sc
            if bool(logm.any()):
                x = torch.where(logm, torch.log1p(x.clamp_min(0.0)), x)
            x = (x - center) * inv
        return x * m.unsqueeze(-1), m


class FeatureBank:
    """The hand-crafted stroke features, windowed over the SAME index as the raw
    bank -- identical windows, identical folds, so the only difference between the
    two models is what a stroke is represented by.

    This is the baseline the raw-sequence encoder has to beat. Without it, a
    mediocre number from the raw model is uninterpretable: it could mean raw
    points are the wrong representation, or it could mean the dataset is hard.
    """

    kind = "features"

    def __init__(self, V, W, y, sess, folds, pre, device=None):
        self.V = np.ascontiguousarray(V, dtype=np.float32)
        self.W = W
        self.y, self.sess, self.folds = y, sess, folds
        self.preproc = pre
        self.device = device or DEVICE

    def __len__(self):
        return len(self.W)

    @property
    def n_channels(self):
        return self.V.shape[1]

    @property
    def max_len(self):
        return None                      # one vector per stroke: no time axis

    def make_encoder(self, mcfg):
        return FeatureWindowEncoder(self.n_channels, mcfg)

    def batch(self, rows):
        x = torch.from_numpy(self.V[self.W[rows]]).to(self.device)    # (b, N, D)
        return x, None                   # no mask: nothing is padded


device: mps


In [15]:
# Masked-out logits / maxima. A hard-coded -1e9 raises outright in float16
# (max ~65504), so anything under torch.autocast on a T4 would die here; taking
# the sentinel from the tensor's own dtype keeps fp16 and fp32 paths identical.
def _neg(t):
    return torch.finfo(t.dtype).min


def _pos(t):
    return torch.finfo(t.dtype).max


class AttnPool(nn.Module):
    """Learned-query attention pooling over one axis, mask-aware."""
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))

    def forward(self, h, mask=None):            # h: (B, N, d), mask: (B, N) bool
        s = self.score(h).squeeze(-1)
        if mask is not None:
            s = s.masked_fill(~mask, _neg(s))
        w = torch.softmax(s, dim=1)
        return (h * w.unsqueeze(-1)).sum(1), w


class ResConvBlock(nn.Module):
    """Dilated residual conv over the time axis, re-masked at both ends."""
    def __init__(self, d, k, dil, p):
        super().__init__()
        self.conv = nn.Conv1d(d, d, k, padding=dil * (k - 1) // 2, dilation=dil)
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(p)

    def forward(self, h, mf):                   # h: (B, d, L), mf: (B, 1, L) float
        z = self.conv(h * mf)                   # zero the padding before it is convolved in
        z = self.norm(z.transpose(1, 2)).transpose(1, 2)
        return (h + self.drop(F.gelu(z))) * mf


class StrokeSeqEncoder(nn.Module):
    """One variable-length stroke of raw points -> one fixed-size vector.

    Two things this has to survive, both properties of the capture and not of the
    user:

    * **Irregular sampling.** The gap between samples is not constant, so an
      ordinary CNN over the position sequence would be reading a distorted clock.
      Rather than resampling onto a uniform grid -- which invents points that
      were never observed and erases the timing jitter that is itself
      identifying -- `dt` is handed in as an input channel. The network is free
      to learn velocity, or anything else, as a function of (dx, dy, dt).
    * **Variable length.** Strokes run from `min_points` to `max_len` samples.
      Padding is masked out of every reduction below, so a 7-point stroke and a
      120-point stroke are both summarised without the padding contributing.

    Pooling is mean+std+max+attention over time: order *within* a stroke is real
    signal (unlike order within a window), but the pooled summary still has to be
    length-invariant.
    """
    def __init__(self, n_ch, mcfg: ModelConfig):
        super().__init__()
        d = mcfg.d_conv
        self.inp = nn.Linear(n_ch, d)
        self.blocks = nn.ModuleList(
            [ResConvBlock(d, mcfg.kernel, dil, mcfg.dropout) for dil in mcfg.dilations])
        self.attn = AttnPool(d)
        self.out = nn.Sequential(
            nn.Linear(4 * d, mcfg.d_stroke), nn.LayerNorm(mcfg.d_stroke), nn.GELU())

    def forward(self, x, m):                    # x: (B, L, C), m: (B, L) bool
        mf = m.unsqueeze(1).to(x.dtype)         # (B, 1, L)
        h = (self.inp(x).transpose(1, 2)) * mf  # (B, d, L)
        for blk in self.blocks:
            h = blk(h, mf)
        h = h.transpose(1, 2)                   # (B, L, d)

        mb = m.unsqueeze(-1)
        n = m.sum(1, keepdim=True).clamp_min(1).to(x.dtype)
        mean = (h * mb).sum(1) / n
        var = (((h - mean.unsqueeze(1)) ** 2) * mb).sum(1) / n
        std = var.clamp_min(1e-8).sqrt()
        mx = h.masked_fill(~mb, _neg(h)).max(1).values
        att, _ = self.attn(h, m)
        return self.out(torch.cat([att, mean, std, mx], dim=-1))


class StrokeSetEncoder(nn.Module):
    """Window of N per-stroke vectors -> one L2-normalised embedding.

    Per-stroke MLP, then pooling over the stroke axis. Pooling is permutation
    invariant by design: within a 25-stroke window the ordering is close to
    arbitrary, and an order-sensitive encoder (GRU/transformer) mostly learns
    session-specific sequencing, which is exactly the thing that does not
    transfer to a new session.
    """
    def __init__(self, d_in, mcfg: ModelConfig):
        super().__init__()
        d, p = mcfg.d_hidden, mcfg.dropout
        layers, prev = [], d_in
        for _ in range(mcfg.n_layers):
            layers += [nn.Linear(prev, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p)]
            prev = d
        self.stroke = nn.Sequential(*layers)
        self.pool_mode = mcfg.pool
        self.attn = AttnPool(d) if "attn" in mcfg.pool else None
        mult = {"mean": 1, "meanstd": 2, "attn+meanstd": 3}[mcfg.pool]
        self.head = nn.Sequential(
            nn.Linear(d * mult, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p),
            nn.Linear(d, mcfg.d_embed),
        )

    def forward(self, x, return_attn=False):     # x: (B, N, d_in)
        h = self.stroke(x)
        parts = [h.mean(1)]
        if "std" in self.pool_mode:
            parts.append(h.std(1, unbiased=False))
        w = None
        if self.attn is not None:
            a, w = self.attn(h)
            parts.insert(0, a)
        z = F.normalize(self.head(torch.cat(parts, dim=-1)), dim=-1)
        return (z, w) if return_attn else z


class RawWindowEncoder(nn.Module):
    """(B, N, L, C) raw points -> (B, d_embed) L2-normalised window embedding.

    Two levels, and the split is deliberate: *inside* a stroke time order matters
    and the conv stack reads it; *across* strokes in a window it does not, and
    the set encoder throws it away.
    """
    def __init__(self, n_ch, mcfg: ModelConfig):
        super().__init__()
        self.n_ch = n_ch
        self.seq = StrokeSeqEncoder(n_ch, mcfg)
        self.set = StrokeSetEncoder(mcfg.d_stroke, mcfg)

    def forward(self, x, m, return_attn=False):
        B, N, L, C = x.shape
        h = self.seq(x.reshape(B * N, L, C), m.reshape(B * N, L)).view(B, N, -1)
        return self.set(h, return_attn=return_attn)


class FeatureWindowEncoder(nn.Module):
    """Baseline encoder: the hand-crafted per-stroke feature vectors straight into
    the same set encoder, same pooling, same loss, same windows.

    It takes (x, m) like RawWindowEncoder and ignores the mask, so both models
    are interchangeable everywhere downstream -- train_siamese, embed, evaluate
    and save_model never branch on which one they were handed.
    """
    def __init__(self, d_in, mcfg: ModelConfig):
        super().__init__()
        self.n_ch = d_in
        self.set = StrokeSetEncoder(d_in, mcfg)

    def forward(self, x, m=None, return_attn=False):
        return self.set(x, return_attn=return_attn)


def pdist(z):
    """Euclidean distance on L2-normalised embeddings == sqrt(2-2cos)."""
    return torch.cdist(z, z, p=2).clamp_min(0)


def batch_hard_triplet(z, labels, margin=0.25):
    """For each anchor: hardest positive, hardest negative, inside the batch.

    margin <= 0 switches to the soft-margin form log1p(exp(dp - dn)), which has
    no threshold to tune and does not go dead once easy triplets are exhausted.
    """
    D = pdist(z)
    same = labels[:, None] == labels[None, :]
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    pos = same & ~eye
    neg = ~same
    valid = pos.any(1) & neg.any(1)
    if not valid.any():
        return z.sum() * 0.0, {}
    dp = (D.masked_fill(~pos, _neg(D))).max(1).values[valid]        # hardest positive
    dn = (D.masked_fill(~neg, _pos(D))).min(1).values[valid]       # hardest negative
    loss = F.relu(dp - dn + margin).mean() if margin > 0 else F.softplus(dp - dn).mean()
    return loss, {"dp": dp.mean().item(), "dn": dn.mean().item(),
                  "viol": (dp + margin > dn).float().mean().item()}


def supcon(z, labels, temperature=0.1):
    """Supervised contrastive loss: softmax over cosine similarities, every
    same-user window in the batch counted as a positive.

    Why this and not triplet. Triplet minimises dp - dn, which is a difference of
    distances and therefore shrinks when the whole embedding shrinks -- collapse
    is a *descent direction*. Here the logits are z_i . z_j / T, and a collapsed
    embedding makes every logit identical, so the softmax becomes uniform and the
    loss hits its maximum log(B-1). The degenerate solution is the worst one
    reachable, not the easiest.

    The temperature sets how hard the ranking is pushed; 0.05-0.2 is the usual
    range, lower being sharper.
    """
    sim = (z @ z.t()) / temperature
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, _neg(sim))
    logprob = sim - torch.logsumexp(sim, dim=1, keepdim=True)

    pos = (labels[:, None] == labels[None, :]) & ~eye
    cnt = pos.sum(1)
    valid = cnt > 0
    if not valid.any():
        return z.sum() * 0.0, {}
    loss = -(logprob * pos).sum(1)[valid] / cnt[valid]

    with torch.no_grad():
        cos = z @ z.t()
        neg = ~pos & ~eye
        sp = cos[pos].mean().item() if pos.any() else float("nan")
        sn = cos[neg].mean().item() if neg.any() else float("nan")
    return loss.mean(), {"cos_pos": sp, "cos_neg": sn, "gap": sp - sn}


def contrastive_pairs(z, labels, margin=1.0):
    """Classic siamese pair loss, kept for comparison against batch_hard."""
    D = pdist(z)
    same = (labels[:, None] == labels[None, :]).float()
    iu = torch.triu_indices(len(z), len(z), offset=1, device=z.device)
    d, s = D[iu[0], iu[1]], same[iu[0], iu[1]]
    loss = (s * d.pow(2) + (1 - s) * F.relu(margin - d).pow(2)).mean()
    return loss, {"d_pos": d[s > 0].mean().item(), "d_neg": d[s == 0].mean().item()}


class PKSampler:
    """P users x K windows per batch -- without it, random batches rarely contain
    a positive pair and the mining above has nothing to mine."""
    def __init__(self, y, sess, P, K, steps, seed=0):
        self.rng = np.random.default_rng(seed)
        self.P, self.K, self.steps = P, K, steps
        self.by_user = {u: np.where(y == u)[0] for u in np.unique(y)}
        self.sess = sess
        self.users = [u for u, v in self.by_user.items() if len(v) >= 2]

    def __iter__(self):
        for _ in range(self.steps):
            P = min(self.P, len(self.users))
            picked = self.rng.choice(self.users, size=P, replace=False)
            batch = []
            for u in picked:
                pool = self.by_user[u]
                # spread the K windows over distinct sessions where possible, so the
                # "hardest positive" is a cross-session pair rather than a neighbour
                sess_u = self.sess[pool]
                order = self.rng.permutation(len(pool))
                seen, first, rest = set(), [], []
                for i in order:
                    (first if sess_u[i] not in seen else rest).append(pool[i])
                    seen.add(sess_u[i])
                take = (first + rest)[: self.K]
                if len(take) < self.K:
                    take = list(self.rng.choice(pool, size=self.K, replace=True))
                batch += take
            yield np.array(batch)

    def __len__(self):
        return self.steps

In [16]:
from sklearn.metrics import roc_curve, auc as _auc


def eer_from_scores(y_true, score):
    """score = similarity (higher == more likely genuine)."""
    fpr, tpr, thr = roc_curve(y_true, score)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2), float(_auc(fpr, tpr)), float(thr[i])


@torch.no_grad()
def embed(enc, bank: "WindowBank", idx=None, bs=64):
    """Embed windows straight out of the bank -- batches are built on demand, so
    the raw points are never materialised per window."""
    enc.eval()
    rows = np.arange(len(bank)) if idx is None else np.asarray(idx)
    out = []
    for i in range(0, len(rows), bs):
        x, m = bank.batch(rows[i:i + bs])
        out.append(enc(x, m).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 1), np.float32)


def templates(z, y):
    """One L2-normalised centroid per user (the 'enrolment' step)."""
    us = np.array(sorted(set(y)))
    T = np.stack([z[y == u].mean(0) for u in us])
    return T / np.linalg.norm(T, axis=1, keepdims=True), us


def template_metrics(z_enroll, y_enroll, z_probe, y_probe, tag=""):
    """Probe each window against every user's enrolment template.

    This is the protocol that matters: enrolment and probe come from different
    sessions by construction, so it cannot be gamed by within-session similarity.
    """
    T, us = templates(z_enroll, y_enroll)
    S = z_probe @ T.T                                   # cosine similarity
    keep = np.isin(y_probe, us)
    S, yp = S[keep], y_probe[keep]
    gt = np.array([list(us).index(u) for u in yp])
    rank1 = float((S.argmax(1) == gt).mean())
    lab = np.zeros_like(S, dtype=int); lab[np.arange(len(gt)), gt] = 1
    eer, auc_, thr = eer_from_scores(lab.ravel(), S.ravel())
    m = {"rank1": rank1, "eer": eer, "auc": auc_, "thr": thr, "n_probe": int(len(yp))}
    if tag:
        print(f"[{tag}] rank-1 {rank1:6.1%}   EER {eer:6.2%}   AUC {auc_:.4f}   "
              f"(n={len(yp)}, {len(us)} users)")
    return m


## 6. Train

Early stopping on **open-set** validation EER: templates from the val users' enrol
session, probes from their probe session, and none of those users in training. So
model selection optimises for strangers rather than for the training identities.

In [18]:
# ---------------------------------------------------------------------------
# Training -- supervised contrastive, open-set validation
# ---------------------------------------------------------------------------


def _cap(rows, cap, seed=0):
    rows = np.asarray(rows)
    if cap and len(rows) > cap:
        return np.sort(np.random.default_rng(seed).choice(rows, cap, replace=False))
    return rows


def train_encoder(bank, ix, mcfg, verbose_every=5):
    """Validation is open-set: templates come from the val users' ENROL session and
    probes from their PROBE session, and none of those users appear in training.
    Early stopping therefore selects for generalisation to strangers rather than
    for memorising the training identities."""
    torch.manual_seed(mcfg.seed); np.random.seed(mcfg.seed)
    y, sess = bank.y, bank.sess
    tr = ix["train"]
    ve, vp = ix.get("val_enrol"), ix.get("val_probe")

    enc = bank.make_encoder(mcfg).to(DEVICE)
    n_par = sum(p.numel() for p in enc.parameters())
    shape = f"{bank.n_channels}ch x {bank.max_len}pts" if bank.max_len else f"{bank.n_channels} features"
    print(f"[model:{bank.kind}] {shape} x {mcfg.n_strokes} strokes -> "
          f"{mcfg.d_embed}d  ({n_par/1e6:.2f}M params)")

    opt = torch.optim.AdamW(enc.parameters(), lr=mcfg.lr, weight_decay=mcfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=mcfg.lr, total_steps=mcfg.epochs * mcfg.steps_per_epoch, pct_start=0.2)

    ytr = y[tr]
    sampler = PKSampler(ytr, sess[tr], mcfg.P_users, mcfg.K_windows,
                        mcfg.steps_per_epoch, mcfg.seed)
    lut = {u: i for i, u in enumerate(sorted(set(ytr)))}
    ve_c = _cap(ve, mcfg.eval_max_windows, mcfg.seed) if ve is not None else None
    vp_c = _cap(vp, mcfg.eval_max_windows, mcfg.seed + 1) if vp is not None else None

    best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(1, mcfg.epochs + 1):
        enc.train(); tot, stats, t0 = 0.0, {}, time.time()
        for b in sampler:
            xb, mb = bank.batch(tr[b])
            lb = torch.as_tensor([lut[u] for u in ytr[b]], device=DEVICE)
            z = enc(xb, mb)
            loss, st = supcon(z, lb, mcfg.temperature)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 5.0)
            opt.step(); sched.step()
            tot += loss.item(); stats = st

        z_e = embed(enc, bank, ve_c, mcfg.embed_bs)
        z_p = embed(enc, bank, vp_c, mcfg.embed_bs)
        m = template_metrics(z_e, y[ve_c], z_p, y[vp_c])
        spread = float(np.linalg.norm(z_p - z_p.mean(0), axis=1).mean())
        hist.append({"epoch": ep, "loss": tot / mcfg.steps_per_epoch, "val_eer": m["eer"],
                     "val_rank1": m["rank1"], "spread": spread,
                     "sec": time.time() - t0, **stats})
        if verbose_every and (ep % verbose_every == 0 or ep == 1):
            print(f"  ep {ep:3d}  loss {hist[-1]['loss']:.4f}  "
                  f"val(open-set) EER {m['eer']:6.2%}  rank-1 {m['rank1']:6.1%}  "
                  f"spread {spread:.3f}  gap {stats.get('gap', float('nan')):+.3f}"
                  f"  ({hist[-1]['sec']:.0f}s)")
        if m["eer"] < best - 1e-4:
            best, bad = m["eer"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()}
        else:
            bad += 1
            if bad >= mcfg.patience:
                print(f"  early stop at epoch {ep} (best val EER {best:.2%})"); break

    if best_state is not None:
        enc.load_state_dict(best_state)
    h = pd.DataFrame(hist)
    print(f"[train:{bank.kind}] best open-set val EER {best:.2%}   "
          f"final spread {h.spread.iloc[-1]:.4f}"
          + ("   <-- COLLAPSED" if h.spread.iloc[-1] < 0.05 else ""))
    return enc, h

In [19]:
# --- folds, windows, bank, train --------------------------------------------
mcfg = ModelConfig(n_strokes=12, stride=3, max_span_s=120.0,
                   d_embed=64, temperature=0.1,
                   P_users=16, K_windows=4,          # 16 users per batch: 120 identities
                   lr=5e-4, epochs=60, steps_per_epoch=80, patience=12, seed=0)

st_f = assign_user_folds(strokes, n_val_users=15, n_test_users=30, seed=0)
W, y, sess, folds, wmeta = build_window_index(st_f, mcfg)
ix = role_index(folds, sess, y)

# the tabular preprocessor is fitted on TRAIN-fold strokes only
tr_rows = np.unique(W[ix["train"]])
pre = fit_preprocessor(st_f.iloc[tr_rows], PREP)
V = apply_preprocessor(st_f, pre).to_numpy(dtype=np.float32)
bank = FeatureBank(V, W, y, sess, folds, pre)
print(f"[bank] {V.shape[0]:,} strokes x {V.shape[1]} features")

enc, hist = train_encoder(bank, ix, mcfg)

            strokes  sessions  users
fold                                
test_enrol     5473        30     30
test_probe     1641        30     30
train         18349       155     75
val_enrol      2962        15     15
val_probe       988        15     15
[windows] 8,990 x 12 strokes | 120 users | median span 10.9s
[windows] per-user: min 38, median 73, max 189
[roles] train: 5,600 win / 75 users | val_enrol: 935 win / 15 users | val_probe: 280 win / 15 users | test_enrol: 1,726 win / 30 users | test_probe: 449 win / 30 users
[roles] no user leak into train, no session leak between enrol and probe
[prep] 60 features (dropped 3 constant/empty, log1p on 22)
[bank] 29,413 strokes x 60 features
[model:features] 60 features x 12 strokes -> 64d  (0.33M params)
  ep   1  loss 3.2816  val(open-set) EER 10.74%  rank-1  76.8%  spread 0.764  gap +0.332  (1s)
  ep   5  loss 2.1690  val(open-set) EER  4.94%  rank-1  91.8%  spread 0.917  gap +0.563  (1s)
  ep  10  loss 1.6319  val(open-set) EER  

## 7. Evaluate

Every number here is on the 30 test users, none of whom appear in training.

`evaluate_openset` reports the per-window result and then sweeps how much probe
activity a decision is allowed to accumulate: k consecutive windows averaged
before scoring. On Balabit this curve was the most decision-relevant output of the
whole project — EER roughly halved between 50 seconds and 7 minutes of
observation. `far@frr5` is the number to quote: at a threshold that wrongly
challenges the real user 5% of the time, how often does an impostor get through?

In [21]:
# ---------------------------------------------------------------------------
# Evaluation -- all of it open-set, on users never seen in training
# ---------------------------------------------------------------------------


def chunk_windows(y, sess, k):
    """Runs of k consecutive windows from one session, non-overlapping."""
    out, n, start = [], len(y), 0
    for i in range(1, n + 1):
        if i == n or sess[i] != sess[start] or y[i] != y[start]:
            seg = np.arange(start, i)
            for j in range(0, len(seg) - k + 1, k):
                out.append(seg[j:j + k])
            start = i
    return out


def operating_points(lab, s, frrs=(0.01, 0.05)):
    fpr, tpr, thr = roc_curve(lab, s)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    out = {"eer": float((fpr[i] + fnr[i]) / 2), "auc": float(_auc(fpr, tpr))}
    for t in frrs:
        k = np.where(fnr <= t)[0]
        out[f"far@frr{int(t*100)}"] = float(fpr[k[0]]) if len(k) else np.nan
    return out


def evaluate_openset(enc, bank, ix, mcfg, ks=(1, 2, 4, 8, 16), tag=""):
    """Enrol the held-out users from their enrol session, verify from their probe
    session, and sweep how much probe activity the decision is allowed to see."""
    y, sess, = bank.y, bank.sess
    e, p = ix["test_enrol"], ix["test_probe"]
    z_e = embed(enc, bank, e, mcfg.embed_bs)
    z_p = embed(enc, bank, p, mcfg.embed_bs)
    T, us = templates(z_e, y[e])

    print(f"\n[{tag}] open-set: {len(us)} unseen users, "
          f"{len(e):,} enrol / {len(p):,} probe windows")
    head = template_metrics(z_e, y[e], z_p, y[p], tag=f"{tag} per-window")

    rows = []
    for k in ks:
        ch = chunk_windows(y[p], sess[p], k)
        if len(ch) < 30:
            continue
        yc = np.array([y[p][c[0]] for c in ch])
        secs = float(np.median([wmeta[p][c[-1], 1] - wmeta[p][c[0], 0] for c in ch]))
        zz = np.stack([z_p[c].mean(0) for c in ch])
        zz /= np.linalg.norm(zz, axis=1, keepdims=True)
        S = zz @ T.T
        keep = np.isin(yc, us)
        lab = (yc[keep][:, None] == us[None, :]).astype(int).ravel()
        m = operating_points(lab, S[keep].ravel())
        m.update({"k": k, "median_s": secs, "n_probes": int(keep.sum()),
                  "rank1": float((S[keep].argmax(1) ==
                                  np.array([list(us).index(u) for u in yc[keep]])).mean())})
        rows.append(m)
    sweep = pd.DataFrame(rows)
    print(f"\n[{tag}] accumulated decisions (unseen users):")
    print(sweep[["k", "median_s", "n_probes", "rank1", "eer", "auc",
                 "far@frr1", "far@frr5"]].to_string(index=False,
                 float_format=lambda v: f"{v:,.3f}"))
    return {"per_window": head, "sweep": sweep, "n_users": int(len(us))}

In [22]:
res = evaluate_openset(enc, bank, ix, mcfg, ks=(1, 2, 4, 8, 16), tag="features")
display(hist.tail(8))


[features] open-set: 30 unseen users, 1,726 enrol / 449 probe windows
[features per-window] rank-1  59.7%   EER 12.64%   AUC 0.9513   (n=449, 30 users)

[features] accumulated decisions (unseen users):
 k  median_s  n_probes  rank1   eer   auc  far@frr1  far@frr5
 1    11.208       449  0.597 0.126 0.951     0.340     0.236
 2    14.049       217  0.604 0.124 0.954     0.325     0.223
 4    20.095        99  0.667 0.113 0.958     0.355     0.248
 8    32.215        43  0.674 0.125 0.962     0.319     0.164
    epoch      loss   val_eer  val_rank1    spread       sec   cos_pos   cos_neg       gap
11     12  1.519122  0.043495   0.914286  0.933945  1.111212  0.789400  0.054374  0.735026
12     13  1.505503  0.050510   0.864286  0.944405  1.062394  0.824867 -0.014739  0.839605
13     14  1.437479  0.060714   0.875000  0.945261  1.087195  0.821275 -0.007010  0.828285
14     15  1.424563  0.060459   0.867857  0.933904  1.210301  0.830121  0.006515  0.823607
15     16  1.409025  0.053954   

In [23]:
# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
import joblib

MODEL_ROOT = OUT_ROOT / "models"


def save_model(outdir, enc, bank, mcfg, metrics, hist, tag, extra=None):
    out = Path(outdir) / tag
    out.mkdir(parents=True, exist_ok=True)
    ck = {"state_dict": enc.state_dict(), "kind": bank.kind, "n_ch": enc.n_ch,
          "model_config": asdict(mcfg)}
    if extra:
        ck.update(extra)
    torch.save(ck, out / "encoder.pt")
    joblib.dump(bank.preproc, out / "preprocessor.joblib")
    pd.DataFrame(hist).to_csv(out / "history.csv", index=False)
    metrics["sweep"].to_csv(out / "sweep.csv", index=False)
    (out / "metrics.json").write_text(json.dumps(
        {"per_window": metrics["per_window"], "n_users": metrics["n_users"]},
        indent=2, default=float))
    print(f"[save] -> {out}")
    return out

In [24]:
save_model(MODEL_ROOT, enc, bank, mcfg, res, hist, tag="sapimouse_features")

[save] -> /Users/albi/Documents/Projects/IBM_hackathon_2026/encoder2/data/models/sapimouse_features
